筛选靶点

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import joblib
import scipy.sparse as sp
import scipy.io

# ==================== 1. 全局配置 ====================
# 你的 WSL 专属路径配置
MODEL_DIR = "/mnt/e/2-8.3-shanda/1-feature/9-Master-raw-Master_Clocks_3"
RAT_DATA_DIR = "/mnt/e/2-8.3-shanda/1-data/2-valid/GSE137869_RAW" 
OUTPUT_DIR = os.path.join(MODEL_DIR, "111-Pan_Tissue_Mechanistic_Analysis")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TISSUE_MAPPING = {
    "Liver": "Liver",
    "Kidney": "Kidney",
    "Skin": "Skin",
    "Limb_Muscle": "Muscle",
    "marrow": "BM"
}

# ==================== 2. 自定义数据加载器 ====================
def load_custom_10x(prefix, dir_path):
    mat_path = os.path.join(dir_path, f"{prefix}_matrix.mtx.gz")
    bc_path = os.path.join(dir_path, f"{prefix}_barcodes.tsv.gz")
    gene_path = os.path.join(dir_path, f"{prefix}_genes.tsv.gz")
    
    if not (os.path.exists(mat_path) and os.path.exists(bc_path) and os.path.exists(gene_path)): 
        return None
        
    adata = ad.AnnData(X=scipy.io.mmread(mat_path).T.tocsr())
    adata.obs_names = pd.read_csv(bc_path, header=None, sep='\t')[0].values
    genes_df = pd.read_csv(gene_path, header=None, sep='\t')
    adata.var_names = genes_df[1].values if genes_df.shape[1] > 1 else genes_df[0].values
    adata.var_names_make_unique()
    return adata

# ==================== 3. 核心挖掘引擎 (严格统计学版) ====================
all_clock_genes = set()  # 收集所有模型中出现过的基因
tissue_rescue_status = {} # 存储每个组织中基因的逆转状态

for mouse_model, rat_tissue in TISSUE_MAPPING.items():
    print(f"\n🔬 正在挖掘 {rat_tissue} 组织的逆转靶点...")
    
    # 1. 加载模型与特征提取
    model_path = os.path.join(MODEL_DIR, f"{mouse_model}_Clock.pkl")
    if not os.path.exists(model_path):
        print(f"  ⚠️ 模型不存在，跳过。")
        continue
    clock_features = joblib.load(model_path)['features']
    
    # 统一将首字母大写，用于跨物种(Mouse -> Rat)匹配
    clock_features_cap = [g.capitalize() for g in clock_features]
    all_clock_genes.update(clock_features_cap)
    
    # 初始化该组织的状态字典
    status_dict = {g: "Not in Model" for g in clock_features_cap}
    
    # 2. 提取大鼠单细胞数据
    mat_files = glob.glob(os.path.join(RAT_DATA_DIR, f"*_{rat_tissue}-*_matrix.mtx.gz"))
    adatas = []
    for file_path in mat_files:
        prefix = os.path.basename(file_path).replace("_matrix.mtx.gz", "") 
        adata_tmp = load_custom_10x(prefix, RAT_DATA_DIR)
        if adata_tmp is None: continue
        parts = prefix.split('_')[1].split('-') 
        if len(parts) >= 3 and parts[2] in ['Y', 'O', 'CR']:
            adata_tmp.obs['Condition'] = parts[2]
            adatas.append(adata_tmp)
            
    if not adatas:
        print(f"  ⚠️ 未找到组别数据，跳过。")
        continue
        
    adata = ad.concat(adatas, join="outer")
    adata.obs_names_make_unique()

    # 3. 极简质控与过滤
    sc.pp.filter_cells(adata, min_genes=200)
    adata = adata[(adata.obs['n_genes_by_counts'] > 500)].copy() if 'n_genes_by_counts' in adata.obs else adata
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.obs['Condition'] = pd.Categorical(adata.obs['Condition'], categories=['Y', 'O', 'CR'], ordered=True)

    # 4. 寻找健壮基因 (防极值/过分稀疏)
    sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
    for g in clock_features_cap:
        if g not in adata.var_names or adata.var.loc[g, 'n_cells_by_counts'] <= (adata.n_obs * 0.03):
            status_dict[g] = "Missing/Low Expr"
            
    robust_genes = [g for g, stat in status_dict.items() if stat != "Missing/Low Expr"]
    
    if len(robust_genes) < 2:
        tissue_rescue_status[rat_tissue] = status_dict
        print(f"  ⚠️ {rat_tissue} 组织表达丰度达标的靶基因不足2个，跳过差异分析。")
        continue

    # =========================================================================
    # 5. 🌟 顶刊级别的严格判定：计算 logFC 与 Adjusted P-value (FDR)
    # =========================================================================
    sc.tl.rank_genes_groups(adata, groupby='Condition', reference='Y', groups=['O'], method='wilcoxon', key_added='DE_aging')
    sc.tl.rank_genes_groups(adata, groupby='Condition', reference='O', groups=['CR'], method='wilcoxon', key_added='DE_treatment')
    
    # 提取完整的数据框（包含 logfoldchanges 和 pvals_adj）
    df_aging = sc.get.rank_genes_groups_df(adata, group='O', key='DE_aging').set_index('names')
    df_treat = sc.get.rank_genes_groups_df(adata, group='CR', key='DE_treatment').set_index('names')

    for g in robust_genes:
        # 安全提取衰老和干预的 logFC
        fc_a = df_aging.loc[g, 'logfoldchanges'] if g in df_aging.index else 0
        fc_t = df_treat.loc[g, 'logfoldchanges'] if g in df_treat.index else 0
        
        # 安全提取衰老和干预的 FDR (调整后 P 值)
        pval_a = df_aging.loc[g, 'pvals_adj'] if g in df_aging.index else 1.0
        pval_t = df_treat.loc[g, 'pvals_adj'] if g in df_treat.index else 1.0
        
        # 🌟 核心逻辑：必须同时满足方向相反且具备显著性！
        if (fc_a * fc_t < 0) and (pval_a < 0.05) and (pval_t < 0.1):
            if fc_a > 0:
                status_dict[g] = "Rescued (Down in CR)"
            else:
                status_dict[g] = "Rescued (Up in CR)"
        elif pval_a < 0.05:
            # 在衰老中显著改变了，但 CR 没能将其显著逆转
            status_dict[g] = "Failed/No Rescue"
        else:
            # 连衰老都没发生显著变化，属于组织特异性不足/统计噪音基因
            status_dict[g] = "Not sig. in Aging"
            
    tissue_rescue_status[rat_tissue] = status_dict
    print(f"  ✅ {rat_tissue} 靶点挖掘完成！")

# ==================== 4. 汇总总表并保存 ====================
print("\n📊 正在整合泛组织全局数据表...")

# 构建 DataFrame，行是所有特征基因，列是各个验证组织
df_summary = pd.DataFrame(index=list(all_clock_genes))

for tissue, status_dict in tissue_rescue_status.items():
    df_summary[tissue] = df_summary.index.map(lambda g: status_dict.get(g, "Not in Model"))

# 增加一列统计信息：该基因在多少个组织中被【显著逆转】？
rescue_markers = ["Rescued (Down in CR)", "Rescued (Up in CR)"]
df_summary['Total_Rescue_Count'] = df_summary.isin(rescue_markers).sum(axis=1)

# 按被逆转的组织数量降序排列，把真正的“万能黄金靶点”顶到最上面
df_summary = df_summary.sort_values(by='Total_Rescue_Count', ascending=False)

# 保存文件
output_file = os.path.join(OUTPUT_DIR, "Pan_Tissue_Golden_Targets_Strict.csv")
df_summary.to_csv(output_file)

print(f"\n✅ 提取完成！全局严格版靶点数据表已保存至:\n👉 {output_file}")
print("🔥 提示：去打开这个 CSV 文件，顶部的基因（Count >= 2）就是你文章中最硬核的机制发现！")

1-棒棒糖图

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# ==================== 1. 主刊级别全局配置 ====================
# 强制使用 Arial 无衬线字体
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'sans-serif']

# ★ 统一微距字号 6pt
mpl.rcParams['font.size'] = 6
mpl.rcParams['axes.titlesize'] = 6
mpl.rcParams['axes.labelsize'] = 6
mpl.rcParams['xtick.labelsize'] = 6
mpl.rcParams['ytick.labelsize'] = 6
mpl.rcParams['legend.fontsize'] = 6

# ★ 极致纤细的线条与边框 0.6pt
mpl.rcParams['axes.linewidth'] = 0.6
mpl.rcParams['xtick.major.width'] = 0.6
mpl.rcParams['ytick.major.width'] = 0.6
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.direction'] = 'out'

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

OUTPUT_FIG_DIR = '/mnt/e/2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR'
os.makedirs(OUTPUT_FIG_DIR, exist_ok=True)

# ==================== 2. 录入真实数据 (仅保留 CR) ====================
data = [
    {'Tissue': 'Bone Marrow', 'R': 0.86, 'Intervention': 'CR'},
    {'Tissue': 'Liver',       'R': 0.84, 'Intervention': 'CR'},
    {'Tissue': 'Muscle',      'R': 0.35, 'Intervention': 'CR'},
    {'Tissue': 'Kidney',      'R': 0.25, 'Intervention': 'CR'},
    {'Tissue': 'Skin',        'R': 0.24, 'Intervention': 'CR'}
]

df = pd.DataFrame(data)
# 确保数据按 R 值从高到低排序 (在Y轴上从下往上画)
df = df.sort_values(by='R', ascending=True).reset_index(drop=True)

# ==========================================
# ★ 3. 核心绝招：绝对物理尺寸 (42mm x 38mm)
# ==========================================
MM2IN = 1 / 25.4
AXES_W = 42 * MM2IN  # 中间黑框宽度：42mm
AXES_H = 38 * MM2IN  # 中间黑框高度：38mm

# 设定页边距 (英寸)
# 注意：左侧边距(l_marg)稍微给大一点(0.7)，因为 Y 轴有像 "Bone Marrow" 这样较长的文字
# 右侧边距(r_marg)留 1.0 给图例
l_marg, r_marg, b_marg, t_marg = 0.7, 1.0, 0.45, 0.35 
fig_w = AXES_W + l_marg + r_marg
fig_h = AXES_H + b_marg + t_marg

print("\n正在生成绝对比例 (42x38mm) 的棒棒糖图...")
fig = plt.figure(figsize=(fig_w, fig_h))
# 绝对定位创建坐标轴：保证黑框精确等于 42x38 mm
ax = fig.add_axes([l_marg/fig_w, b_marg/fig_h, AXES_W/fig_w, AXES_H/fig_h])

# 颜色设置 (仅保留 CR 的经典冷蓝)
palette = {'CR': '#4DBBD5'}

# 1. 画棒棒糖的“棍子” (水平线)
# linewidth 降至 1.0pt，纤细高级
for index, row in df.iterrows():
    ax.hlines(y=index, xmin=0, xmax=row['R'], 
              color=palette[row['Intervention']], alpha=0.9, linewidth=1.0, zorder=1)

# 2. 画棒棒糖的“糖果” (散点)
# 散点面积 (s) 缩减至 25，搭配极细 (0.6pt) 的白边
sns.scatterplot(
    data=df, 
    x='R', 
    y='Tissue', 
    hue='Intervention',
    palette=palette,
    s=25, 
    edgecolor='white',
    linewidth=0.6,
    zorder=3,
    ax=ax
)

# 3. 添加竖向参考线 (阈值分割线)
ax.axvline(x=0.5, color='grey', linestyle='--', linewidth=0.6, alpha=0.8, zorder=0)
# 注释文字：去除 bold，字号 5pt
ax.text(0.52, 0.5, "Strong responders (|R| > 0.5)", color='#555555', 
        fontsize=5, fontstyle='italic', va='center')

# 4. 图表修饰
ax.set_title("Organ Heterogeneity in Transcriptomic Reshaping", pad=8, fontweight='normal')
ax.set_xlabel("Transcriptomic Reshaping Magnitude (|Pearson R|)")
ax.set_ylabel("") 

# 优化X轴范围，留出右侧少许呼吸空间
ax.set_xlim(0, 1.0)

# 强制开启四周纯黑边框，线宽 0.6，并去除网格线
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.6)
ax.tick_params(direction='out', length=3.0, width=0.6, colors='black')
ax.grid(False)

# 优化图例：移至外部，无边框
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, title='Intervention', title_fontsize=6,
          bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)

# ==================== 4. 保存输出 ====================
# ★ 绝对禁止使用 plt.tight_layout() 或 bbox_inches='tight'，否则会破坏 42x38mm 的比例
save_path = os.path.join(OUTPUT_FIG_DIR, "Figure_7_Organ_Heterogeneity_Forest_Plot.pdf")
plt.savefig(save_path)
plt.close(fig)

print(f"✅ 主刊级器官异质性森林图 (绝对比例版) 已生成！\n👉 路径: {save_path}")

2-基因表达逆转散点图

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.stats import pearsonr, ttest_ind
import joblib
import scipy.sparse as sp
import scipy.io

# ==================== 1. 主刊级别全局配置 ====================
# ★ 第一步：让 Scanpy 初始化，防止覆盖后续自定义
sc.settings.set_figure_params(dpi=300, facecolor='white', format='pdf')

# ★ 第二步：强制注入极限微距排版配置
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'sans-serif']

# 统一将所有层级的字号降至 6pt (拒绝粗体和大字号)
mpl.rcParams['font.size'] = 6
mpl.rcParams['axes.titlesize'] = 6
mpl.rcParams['axes.labelsize'] = 6
mpl.rcParams['xtick.labelsize'] = 6
mpl.rcParams['ytick.labelsize'] = 6
mpl.rcParams['legend.fontsize'] = 6

# 全局线宽保持极细的 0.6pt
mpl.rcParams['axes.linewidth'] = 0.6
mpl.rcParams['xtick.major.width'] = 0.6
mpl.rcParams['ytick.major.width'] = 0.6
mpl.rcParams['xtick.direction'] = 'out'
mpl.rcParams['ytick.direction'] = 'out'

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

# 【路径配置】请确保这里是你的真实路径
MODEL_DIR = "/mnt/e/2-8.3-shanda/1-feature/9-Master-raw-Master_Clocks_3"
RAT_DATA_DIR = "/mnt/e/2-8.3-shanda/1-data/2-valid/GSE137869_RAW" 

OUTPUT_FIG_DIR = "/mnt/e/2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR"
os.makedirs(OUTPUT_FIG_DIR, exist_ok=True)
sc.settings.figdir = OUTPUT_FIG_DIR

# ==================== 2. 核心战术：组织映射字典 ====================
TISSUE_MAPPING = {
    "Liver": "Liver",
    "Kidney": "Kidney",
    "Skin": "Skin",
    "Limb_Muscle": "Muscle",
    "marrow": "BM",
}

# ==================== 3. 核心辅助函数 ====================
def load_custom_10x(prefix, dir_path):
    mat_path, bc_path, gene_path = [os.path.join(dir_path, f"{prefix}_{x}") for x in ["matrix.mtx.gz", "barcodes.tsv.gz", "genes.tsv.gz"]]
    if not (os.path.exists(mat_path) and os.path.exists(bc_path) and os.path.exists(gene_path)): return None
    adata = ad.AnnData(X=scipy.io.mmread(mat_path).T.tocsr())
    adata.obs_names = pd.read_csv(bc_path, header=None, sep='\t')[0].values
    genes_df = pd.read_csv(gene_path, header=None, sep='\t')
    adata.var_names = genes_df[1].values if genes_df.shape[1] > 1 else genes_df[0].values
    adata.var_names_make_unique()
    return adata

# ==========================================
# ★ 核心绝招：绝对物理尺寸 (42mm x 38mm)
# ==========================================
MM2IN = 1 / 25.4
AXES_W = 42 * MM2IN  # 中间黑框宽度：42mm
AXES_H = 38 * MM2IN  # 中间黑框高度：38mm

def set_closed_box(ax):
    """强制开启四周纯黑边框，线宽 0.6，并去除网格线"""
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.6)
    ax.tick_params(direction='out', length=3.0, width=0.6, colors='black')
    ax.grid(False)

def add_stat_bracket(ax, x1, x2, y, h, p_val):
    """精准复刻倒 U 型括号标注，直接标注精确 P 值"""
    ax.plot([x1, x1, x2, x2], [y-h, y, y, y-h], lw=0.6, c='black') 
    
    if np.isnan(p_val):
        p_str = "ns"
    elif p_val < 2.2e-16:
        p_str = "< 2.2e-16"
    else:
        p_str = f"= {p_val:.1e}"
        
    stat_text = f"P {p_str}" if p_str != "ns" else "ns"
    ax.text((x1+x2)*.5, y + h*0.1, stat_text, ha='center', va='bottom', color='black', fontsize=6)

# ==========================================
# 主流程函数
# ==========================================
def process_single_tissue(mouse_model_name, rat_tissue_name):
    print(f"\n" + "="*50)
    print(f"🚀 开始执行跨物种验证: 小鼠 [{mouse_model_name}] -> 大鼠 [{rat_tissue_name}]")
    print("="*50)
    
    # 1. 加载模型
    model_path = os.path.join(MODEL_DIR, f"{mouse_model_name}_Clock.pkl")
    if not os.path.exists(model_path):
        print(f"⚠️ 找不到模型: {model_path}，跳过该组织。")
        return
    model_package = joblib.load(model_path)
    master_model = model_package['model'] 
    clock_features = model_package['features']
    
    # 2. 提取大鼠数据
    mat_files = glob.glob(os.path.join(RAT_DATA_DIR, f"*_{rat_tissue_name}-*_matrix.mtx.gz"))
    adatas = []
    for file_path in mat_files:
        prefix = os.path.basename(file_path).replace("_matrix.mtx.gz", "") 
        adata_tmp = load_custom_10x(prefix, RAT_DATA_DIR)
        if adata_tmp is None: continue
        parts = prefix.split('_')[1].split('-') 
        if len(parts) >= 3 and parts[2] in ['Y', 'O', 'CR']:
            adata_tmp.obs['Condition'] = parts[2]
            adatas.append(adata_tmp)
            
    if not adatas:
        print(f"⚠️ 未能找到大鼠 {rat_tissue_name} 的数据，跳过。")
        return
        
    adata = ad.concat(adatas, join="outer")
    adata.obs_names_make_unique()

    # 3. 质控与过滤
    adata.var['mt'] = adata.var_names.str.startswith('mt-') | adata.var_names.str.startswith('Mt-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
    sc.pp.filter_cells(adata, min_genes=200)
    adata = adata[(adata.obs['n_genes_by_counts'] > 500)].copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.obs['Condition'] = pd.Categorical(adata.obs['Condition'], categories=['Y', 'O', 'CR'], ordered=True)

    # 盲测年龄预测
    X_target = np.zeros((adata.n_obs, len(clock_features)))
    for i, gene in enumerate(clock_features):
        rat_gene = gene.capitalize()
        if rat_gene in adata.var_names:
            expr = adata[:, rat_gene].X
            X_target[:, i] = expr.toarray().flatten() if sp.issparse(expr) else expr.flatten()
        elif gene in adata.var_names:
            expr = adata[:, gene].X
            X_target[:, i] = expr.toarray().flatten() if sp.issparse(expr) else expr.flatten()
            
    adata.obs['Predicted_Age'] = master_model.predict(X_target)

    # =========================================================
    # ★ 绘制绝对物理尺寸 (42x38mm) 的提琴图
    # =========================================================
    l_marg, r_marg, b_marg, t_marg = 0.55, 0.2, 0.45, 0.35 
    fig_w = AXES_W + l_marg + r_marg
    fig_h = AXES_H + b_marg + t_marg

    fig_v = plt.figure(figsize=(fig_w, fig_h))
    ax_v = fig_v.add_axes([l_marg/fig_w, b_marg/fig_h, AXES_W/fig_w, AXES_H/fig_h])
    
    sns.violinplot(x='Condition', y='Predicted_Age', data=adata.obs, 
                   palette=['#66C2A5', '#FC8D62', '#8DA0CB'], 
                   inner=None, linewidth=0.5, ax=ax_v, saturation=1.0)
    
    # 内部箱线图细腻风格
    sns.boxplot(x='Condition', y='Predicted_Age', data=adata.obs, 
                color='white', width=0.12, fliersize=0, zorder=2, ax=ax_v, 
                boxprops={'edgecolor':'black','linewidth':0.6},
                medianprops={'color':'black', 'linewidth':0.8}, 
                whiskerprops={'color':'black', 'linewidth':0.6}, 
                capprops={'color':'black', 'linewidth':0.6})

    y_vals = adata.obs[adata.obs['Condition'] == 'Y']['Predicted_Age']
    o_vals = adata.obs[adata.obs['Condition'] == 'O']['Predicted_Age']
    cr_vals = adata.obs[adata.obs['Condition'] == 'CR']['Predicted_Age']

    y_median = y_vals.median()
    ax_v.axhline(y_median, color='grey', linestyle='--', linewidth=0.6, alpha=0.8, zorder=0)

    _, p_aging = ttest_ind(o_vals, y_vals, alternative='greater', equal_var=False)
    _, p_treat = ttest_ind(cr_vals, o_vals, alternative='less', equal_var=False)

    y_top = adata.obs['Predicted_Age'].quantile(0.99)
    y_bottom = adata.obs['Predicted_Age'].quantile(0.01) - 1.0
    h_offset = (y_top - y_bottom) * 0.05

    add_stat_bracket(ax_v, 0, 1, y_top + h_offset*3, h_offset*1.5, p_aging)
    add_stat_bracket(ax_v, 1, 2, y_top + h_offset*6, h_offset*1.5, p_treat)

    ax_v.set_ylim(y_bottom, y_top + h_offset * 11)
    delta_age = o_vals.mean() - cr_vals.mean()
    
    ax_v.set_title(f"Rat {rat_tissue_name} (Δ Age = -{delta_age:.2f})", pad=8, fontweight='normal')
    ax_v.set_ylabel("Predicted biological age")
    ax_v.set_xticklabels(['Y', 'O', 'CR'])
    ax_v.set_xlabel("")
    
    set_closed_box(ax_v)
    
    # 绝对禁止 bbox_inches='tight' 以保护方框比例
    plt.savefig(os.path.join(OUTPUT_FIG_DIR, f"{rat_tissue_name}_0_Violin.pdf"))
    plt.close(fig_v)
    print(f"   -> 提琴图已生成: {rat_tissue_name}_0_Violin.pdf")


    # =========================================================
    # 4. 特征对齐与防极值清洗 (为散点图做准备)
    # =========================================================
    available_genes = [g.capitalize() if g.capitalize() in adata.var_names else g for g in clock_features if g.capitalize() in adata.var_names or g in adata.var_names]
    sc.pp.calculate_qc_metrics(adata, percent_top=None, log1p=False, inplace=True)
    robust_genes = [g for g in available_genes if adata.var.loc[g, 'n_cells_by_counts'] > (adata.n_obs * 0.03)]
    
    if len(robust_genes) < 3:
        print(f"⚠️ {rat_tissue_name} 保留的稳健基因太少({len(robust_genes)}个)，无法绘制散点图，跳过。")
        return

    # 5. 生成散点图 (绝对定位 42x38mm)
    sc.tl.rank_genes_groups(adata, groupby='Condition', reference='Y', groups=['O'], method='wilcoxon', key_added='DE_aging')
    sc.tl.rank_genes_groups(adata, groupby='Condition', reference='O', groups=['CR'], method='wilcoxon', key_added='DE_treatment')
    logfc_aging = sc.get.rank_genes_groups_df(adata, group='O', key='DE_aging').set_index('names')['logfoldchanges']
    logfc_treat = sc.get.rank_genes_groups_df(adata, group='CR', key='DE_treatment').set_index('names')['logfoldchanges']

    df_plot = pd.DataFrame({'logFC_Aging': logfc_aging[robust_genes], 'logFC_Treatment': logfc_treat[robust_genes]}).dropna()
    rescued = (df_plot['logFC_Aging'] * df_plot['logFC_Treatment'] < 0)

    # 散点图如果没有外部图例，右侧边距可以小一点
    fig_s = plt.figure(figsize=(fig_w, fig_h))
    ax_s = fig_s.add_axes([l_marg/fig_w, b_marg/fig_h, AXES_W/fig_w, AXES_H/fig_h])
    
    colors = ['#D95F02' if r else 'grey' for r in rescued]
    ax_s.scatter(df_plot['logFC_Aging'], df_plot['logFC_Treatment'], c=colors, alpha=0.9, s=15, edgecolor='white', linewidth=0.3)
    
    ax_s.axhline(0, color='grey', linestyle='dashed', lw=0.6, zorder=0)
    ax_s.axvline(0, color='grey', linestyle='dashed', lw=0.6, zorder=0)
    
    r, p_corr = pearsonr(df_plot['logFC_Aging'], df_plot['logFC_Treatment'])
    p_corr_str = "< 2.2e-16" if p_corr < 2.2e-16 else f"= {p_corr:.1e}"
    
    sns.regplot(x='logFC_Aging', y='logFC_Treatment', data=df_plot, scatter=False, 
                color='black', ax=ax_s, line_kws={'linestyle':'--', 'lw':0.8}, ci=95, seed=42)
    
    ax_s.set_title(f"Pearson R = {r:.2f}, P {p_corr_str}", pad=8, fontweight='normal')
    ax_s.set_xlabel(r"Aging Effect: $\log_2$(O / Y)")
    ax_s.set_ylabel(r"Treatment Effect: $\log_2$(CR / O)")
    
    set_closed_box(ax_s)
    plt.savefig(os.path.join(OUTPUT_FIG_DIR, f"{rat_tissue_name}_1_Scatter.pdf"))
    plt.close(fig_s)
    print(f"   -> 散点图已生成: {rat_tissue_name}_1_Scatter.pdf")

    # 6. 生成重塑热图
    with mpl.rc_context({'font.size': 6, 'figure.figsize': [2.8, 3.2]}):
        sc.pl.matrixplot(adata, var_names=robust_genes, groupby='Condition', 
                         dendrogram=True, cmap='RdBu_r', standard_scale='var',
                         vmin=0, vmax=1,
                         colorbar_title='Scaled Expr', title="",
                         save=f"_{rat_tissue_name}_2_Heatmap.pdf", show=False)
        plt.close()
        
    print(f"✅ {rat_tissue_name} 验证完成！图片已保存至 {OUTPUT_FIG_DIR}")

# ==================== 4. 启动主循环批处理 ====================
for mouse_model, rat_tissue in TISSUE_MAPPING.items():
    try:
        process_single_tissue(mouse_model, rat_tissue)
    except Exception as e:
        print(f"❌ 处理 {rat_tissue} 时发生报错: {e}")

print("\n🎉 全部组织跨物种验证执行完毕！请前往目标文件夹验收结果！")

3-CR-靶点基因表达

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
from scipy.stats import ttest_ind
import scipy.io
import warnings
warnings.filterwarnings("ignore")

# ==================== 1. 强力清除环境劫持 ====================
sns.reset_orig() 

# 初始化 Scanpy 设置
sc.settings.set_figure_params(dpi=300, facecolor='white')

# ★ 关键：解决 AI 字母分离问题
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['svg.fonttype'] = 'none' 

mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'sans-serif']
mpl.rcParams['mathtext.default'] = 'regular' 

# ★ 关键：强力关闭 Scanpy 默认开启的背景网格线
mpl.rcParams['axes.grid'] = False

# 严格遵照指令：全局所有字号锁定为 4.5pt
mpl.rcParams['font.size'] = 4.5
mpl.rcParams['axes.titlesize'] = 4.5
mpl.rcParams['axes.labelsize'] = 4.5
mpl.rcParams['xtick.labelsize'] = 4.5
mpl.rcParams['ytick.labelsize'] = 4.5
mpl.rcParams['legend.fontsize'] = 4.5

# 极致纤细线条 0.5pt
mpl.rcParams['axes.linewidth'] = 0.5
mpl.rcParams['xtick.major.width'] = 0.5
mpl.rcParams['ytick.major.width'] = 0.5

# ==================== 2. 路径与配置 ====================
RAT_DATA_DIR = "/mnt/e/2-8.3-shanda/1-data/2-valid/GSE137869_RAW" 
OUTPUT_DIR = "/mnt/e/2-8.3-shanda/1-feature/1-figure/0-4-result-5-CR-leipameisu/3-CR"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 颜色配置 (莫兰迪色系)
PALETTE = {'Y': '#61A48F', 'O': '#EA8D74', 'CR': '#3C5488'} 

TARGET_PAIRS = {
    "Liver": ["Ctsl"], "Kidney": ["Rps10"], "Muscle": ["Ctsl"],
    "Skin": ["Hspa8", "Rps28", "S100a6"], "BM": ["Rps10", "Hspa8", "Rps28", "S100a6"]
}

# 按照指定基因顺序排列
PLOT_PAIRS = [
    ("Ctsl", "Liver"),  ("Hspa8", "Skin"), ("Rps10", "Kidney"), ("Rps28", "Skin"), ("S100a6", "Skin"),
    ("Ctsl", "Muscle"), ("Hspa8", "BM"),   ("Rps10", "BM"),     ("Rps28", "BM"),   ("S100a6", "BM")
]

# ==================== 3. 数据加载函数 ====================
def load_custom_10x(prefix, dir_path):
    mat_path, bc_path, gene_path = [os.path.join(dir_path, f"{prefix}_{x}") for x in ["matrix.mtx.gz", "barcodes.tsv.gz", "genes.tsv.gz"]]
    if not (os.path.exists(mat_path) and os.path.exists(bc_path) and os.path.exists(gene_path)): return None
    adata = ad.AnnData(X=scipy.io.mmread(mat_path).T.tocsr())
    adata.obs_names = pd.read_csv(bc_path, header=None, sep='\t')[0].values
    genes_df = pd.read_csv(gene_path, header=None, sep='\t')
    adata.var_names = genes_df[1].values if genes_df.shape[1] > 1 else genes_df[0].values
    adata.var_names_make_unique()
    return adata

plot_data_list = []
print("🚀 开始提取单细胞数据...")
for tissue, genes in TARGET_PAIRS.items():
    mat_files = glob.glob(os.path.join(RAT_DATA_DIR, f"*_{tissue}-*_matrix.mtx.gz"))
    adatas = []
    for file_path in mat_files:
        prefix = os.path.basename(file_path).replace("_matrix.mtx.gz", "") 
        adata_tmp = load_custom_10x(prefix, RAT_DATA_DIR)
        if adata_tmp is None: continue
        parts = prefix.split('_')[1].split('-') 
        if len(parts) >= 3 and parts[2] in ['Y', 'O', 'CR']:
            adata_tmp.obs['Condition'] = parts[2]
            adatas.append(adata_tmp)
    if not adatas: continue
    adata = ad.concat(adatas, join="outer")
    adata.obs_names_make_unique()
    sc.pp.filter_cells(adata, min_genes=200)
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    for gene in genes:
        gene_cap = gene.capitalize()
        if gene_cap in adata.var_names:
            expr = adata[:, gene_cap].X.toarray().flatten() if hasattr(adata[:, gene_cap].X, 'toarray') else adata[:, gene_cap].X.flatten()
            plot_data_list.append(pd.DataFrame({'Expression': expr, 'Condition': adata.obs['Condition'].values, 'Tissue': tissue, 'Gene': gene_cap}))

if not plot_data_list: sys.exit("❌ 数据提取失败。")
df_all = pd.concat(plot_data_list, ignore_index=True)
df_all['Condition'] = pd.Categorical(df_all['Condition'], categories=['Y', 'O', 'CR'], ordered=True)
df_all = df_all[df_all['Expression'] > 0] 

# ==================== 4. P值格式化 ====================
def format_main_p(p):
    if p < 2.2e-16: return r"P < 2.2e-16" 
    else:
        sci_str = f"{p:.1e}"
        base, exp = sci_str.split('e')
        return f"P = {base}e{int(exp)}"

def add_stat_bracket(ax, x1, x2, y_base, h_bar, p_val, alpha=0.05):
    ax.plot([x1, x1, x2, x2], [y_base, y_base+h_bar, y_base+h_bar, y_base], lw=0.4, c='black', zorder=5) 
    text = format_main_p(p_val) if p_val < alpha else "ns"
    ax.text((x1+x2)*.5, y_base + h_bar + (h_bar*0.1), text, ha='center', va='bottom', color='black', fontsize=4.5, zorder=6) 

# ==================== 5. 开始绘图 ====================
print("🎨 正在生成纯净无网格的 4.5pt 矩阵排版...")

width_inch = 105 / 25.4
height_inch = 65 / 25.4
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(width_inch, height_inch))
axes_flat = axes.flatten()

for i, (gene, tissue) in enumerate(PLOT_PAIRS):
    ax = axes_flat[i]
    df_sub = df_all[(df_all['Gene'] == gene) & (df_all['Tissue'] == tissue)]
    if df_sub.empty:
        ax.set_visible(False)
        continue
    
    # ★ 强制关闭该子图的任何背景网格线
    ax.grid(False)
    
    # 小提琴本体
    sns.violinplot(
        data=df_sub, x='Condition', y='Expression', palette=PALETTE, 
        order=['Y', 'O', 'CR'], linewidth=0.4, inner=None, cut=0, ax=ax, saturation=1, zorder=1
    )
    
    # 嵌套箱线图 (白芯黑边)
    sns.boxplot(
        data=df_sub, x='Condition', y='Expression', order=['Y', 'O', 'CR'],
        color='white', width=0.12, fliersize=0, zorder=2, ax=ax,
        boxprops={'facecolor':'white', 'edgecolor':'black', 'linewidth':0.4},
        medianprops={'color':'black', 'linewidth':0.8},
        whiskerprops={'color':'black', 'linewidth':0.4},
        capprops={'color':'black', 'linewidth':0.4}
    )
    
    y_expr = df_sub[df_sub['Condition'] == 'Y']['Expression']
    o_expr = df_sub[df_sub['Condition'] == 'O']['Expression']
    cr_expr = df_sub[df_sub['Condition'] == 'CR']['Expression']
    
    y_max = df_sub['Expression'].max()
    h_bar = y_max * 0.03 
    
    # 年轻组中位数基准线 (唯一保留的横线)
    if len(y_expr) > 0:
        y_median = y_expr.median()
        ax.axhline(y_median, color='grey', linestyle='--', linewidth=0.5, alpha=0.7, zorder=0)

    # 统计 P 值
    if len(y_expr)>3 and len(o_expr)>3 and len(cr_expr)>3:
        _, p_aging = ttest_ind(o_expr, y_expr, equal_var=False)
        _, p_treat = ttest_ind(cr_expr, o_expr, equal_var=False)
        
        y_base_aging = y_max + y_max * 0.05
        y_base_treat = y_max + y_max * 0.25 
        
        add_stat_bracket(ax, 0, 1, y_base_aging, h_bar, p_aging)
        add_stat_bracket(ax, 1, 2, y_base_treat, h_bar, p_treat)
        ax.set_ylim(0, y_base_treat + h_bar*6)

    # 边框设置
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.spines['left'].set_linewidth(0.5)
    
    # 标题 (使用纯文本减少 AI 解析错误)
    ax.set_title(f"{gene} in {tissue}", fontsize=4.5, pad=3, fontstyle='italic')
    
    if i % 5 == 0:
        ax.set_ylabel("log1p Expression", labelpad=2, fontsize=4.5)
    else:
        ax.set_ylabel("")
        
    ax.set_xlabel("")
    ax.set_xticks([0, 1, 2])
    ax.set_xticklabels(['Y', 'O', 'CR'], fontsize=4.5)
    
    ax.tick_params(axis='both', labelsize=4.5, width=0.5, length=2)
    
    if ax.get_legend() is not None: ax.get_legend().remove()

# ==================== 6. 图例与保存 ====================
legend_elements = [
    mpl.patches.Patch(facecolor=PALETTE['Y'], edgecolor='black', linewidth=0.5, label='Young'),
    mpl.patches.Patch(facecolor=PALETTE['O'], edgecolor='black', linewidth=0.5, label='Old'),
    mpl.patches.Patch(facecolor=PALETTE['CR'], edgecolor='black', linewidth=0.5, label='CR')
]

fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 1.08),
           ncol=3, frameon=False, columnspacing=1.5, prop={'size': 4.5})

plt.subplots_adjust(wspace=0.3, hspace=0.5, bottom=0.1, top=0.85) 

# ★ 保存 PDF 文件 (文字无断裂)
save_pdf = os.path.join(OUTPUT_DIR, "Figure_8_4.5pt_NoGrid_Violins.pdf")
plt.savefig(save_pdf, dpi=300, bbox_inches='tight')
plt.show(fig)

print(f"✅ 完美！恼人的背景网格线已全部清理干净，图表已导出至:\n👉 {save_pdf}")